# BinderDiffuser walkthrough: PD-L1 binder design

This notebook walks through the full BinderDiffuser pipeline against PD-L1
(IgV domain, chain B of PDB 5O45). The CPU-friendly stages run inline; the
two GPU-bound stages (RFdiffusion and ColabFold) are previewed via dry-run
and pointed to a Colab notebook for execution.

**Sections**
1. Load target PDB and inspect chains
2. Resolve motif residues into RFdiffusion contig segments
3. Build a `MotifSpec` and preview contig strings
4. Inspect the metric formulas (scRMSD, scTM, ipTM)
5. Show how filters and ranking compose the final summary table

In [ ]:
from pathlib import Path
from binderdiffuser.config import load_config_from_yaml
from binderdiffuser.targets import resolve_target_motif, list_chains, load_structure
from binderdiffuser.diffusion.motif_spec import MotifSpec, build_contig_string
from binderdiffuser.diffusion.rfdiff_wrapper import RFDiffusionRunner

cfg_path = Path('../examples/pdl1_binder/config.yaml')
cfg = load_config_from_yaml(cfg_path)
print(cfg.run_name, '->', cfg.output_dir)

## 1. Inspect the target structure

`5O45` ships PD-L1 (chain B) and an antibody Fv (chains H/L). For binder
design we keep only chain B; the antibody contact residues become our
starting motif because they sit on the PD-1-competing face.

In [ ]:
structure = load_structure(cfg.target.pdb_path)
print('chains in target PDB:', list_chains(structure))

## 2. Motif residues -> contiguous segments

`extract_motif_segments` turns the residue list `[56..65]` into a tuple of
`MotifSegment(chain='B', start=56, end=65)`. If we'd asked for a discontinuous
motif (e.g. residues `[56..58, 80..82]`) it would emit two segments.

In [ ]:
motif = resolve_target_motif(
    cfg.target.pdb_path,
    cfg.target.target_chain,
    cfg.target.motif_residues,
)
for seg in motif.segments:
    print('segment:', seg.as_rfdiff_token(), '(length', seg.length, ')')
print('total motif residues:', motif.total_residues)

## 3. Preview contig strings

Each call to `build_contig_string` produces one realization of binder length
and flank sizes. We sample a few and inspect the diversity.

In [ ]:
spec = MotifSpec(
    target_motif=motif,
    binder_length_min=cfg.diffusion.binder_length_min,
    binder_length_max=cfg.diffusion.binder_length_max,
)
runner = RFDiffusionRunner(check_executable=False)
for i, (contig, seed) in enumerate(runner.sample_contigs(spec, num_designs=5, seed=cfg.seed)):
    print(f'design {i:02d}  seed={seed:<10}  contig={contig}')

## 4. Self-consistency metrics

After RFdiffusion + MPNN + ColabFold finish, every (backbone, sequence)
pair is scored on:

- **scRMSD** — Cα-RMSD between RFdiffusion backbone and AF prediction
- **scTM** — TM-score of the same alignment
- **mean pLDDT** — overall AF confidence
- **ipTM** — interface predicted TM (multimer mode)
- **pAE_interface** — mean predicted aligned error across binder/target

See `binderdiffuser.validation.metrics` for the formulas and unit tests.

In [ ]:
from binderdiffuser.validation.metrics import kabsch_rmsd, tm_score
import numpy as np

# Sanity check: identical coordinates -> perfect TM-score
p = np.random.default_rng(0).normal(size=(80, 3))
_, p_aligned = kabsch_rmsd(p, p.copy())
print('TM-score (identity):', tm_score(p, p_aligned, ref_length=80))

## 5. Filtering and ranking

`filter_designs` enforces the hard thresholds in `FilterConfig`; `rank_designs`
sorts the remainder by the composite score (defined in `validation/filters.py`)
and truncates to top-K. The resulting `DesignRecord` list is what gets
serialized to `designs.csv` and what powers the README scatter plot.

In [ ]:
from binderdiffuser.validation.filters import composite_score, DesignRecord
import inspect

print('composite_score weights:')
print(inspect.getsource(composite_score))

## Next: run the GPU stages

Run `binderdiffuser run examples/pdl1_binder/config.yaml` inside the Docker
image (`docker/Dockerfile`) or open `notebooks/03_colab_inference.ipynb`
to execute on a free Colab T4.